In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import mlflow

from mlflow.models import infer_signature

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import average_precision_score

from churn.config import GOLD_TABLE, ID, TARGET, RANDOM_SEED, MLFLOW_EXPERIMENT_NAME
from churn.validation import validar_gold
from churn.pipeline import construir_pipeline

usuario = spark.sql("SELECT current_user()").collect()[0][0]
experiment = mlflow.set_experiment(f"/Users/{usuario}/{MLFLOW_EXPERIMENT_NAME}")

In [0]:
print(mlflow.__version__)

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

X = pdf.drop(columns=[ID, TARGET])
y = pdf[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

assert np.abs(y_train.mean() - y_test.mean()) < 0.01, "La proporción de churn en train y test no es la misma"

## Experiment

In [0]:
print(experiment.experiment_id)
print(experiment.name)

In [0]:
pipe = construir_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))

with mlflow.start_run(run_name="logreg_base") as run:
    mlflow.log_params({
        "modelo": "LogisticRegression",
        "max_iter": 1000,
        "class_weight": "none",
    })

    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision")
    mlflow.log_metric("pr_auc_cv_media", scores.mean())
    mlflow.log_metric("pr_auc_cv_std", scores.std())

    pipe.fit(X_train, y_train)

    ejemplo = X_train.head(10)
    firma = infer_signature(ejemplo, pipe.predict_proba(ejemplo))

    info_modelo = mlflow.sklearn.log_model(
        sk_model=pipe,
        name="modelo",
        signature=firma,
        input_example=ejemplo,
        pyfunc_predict_fn="predict_proba",
    )

print(info_modelo.model_uri)

In [0]:
modelo_cargado = mlflow.pyfunc.load_model(info_modelo.model_uri)

proba_mlflow = np.asarray(modelo_cargado.predict(X_test))
proba_original = pipe.predict_proba(X_test)

print(proba_mlflow[:3])

assert np.allclose(proba_mlflow, proba_original), \
    "El modelo cargado no reproduce las probabilidades del original"

In [0]:
# SPA: Función para crear una run
# ENG: Function to create a run

def entrenar_y_registrar(modelo, nombre_run, params):
    pipe = construir_pipeline(modelo)

    with mlflow.start_run(run_name=nombre_run) as run:
        mlflow.log_params(params)

        scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                                 scoring="average_precision")
        mlflow.log_metric("pr_auc_cv_media", scores.mean())
        mlflow.log_metric("pr_auc_cv_std", scores.std())

        pipe.fit(X_train, y_train)
        ejemplo = X_train.head(5)
        firma = infer_signature(ejemplo, pipe.predict_proba(ejemplo))
        info = mlflow.sklearn.log_model(
            sk_model=pipe, name="modelo", signature=firma,
            input_example=ejemplo, pyfunc_predict_fn="predict_proba",
        )

    return run.info.run_id, info.model_uri, scores

In [0]:
resultados = {
    "logreg_base": (run.info.run_id, info_modelo.model_uri, scores)
    }

print(resultados.keys())
print(resultados["logreg_base"][2].mean())

In [0]:
from sklearn.ensemble import RandomForestClassifier

resultados["random_forest"] = entrenar_y_registrar(
    RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    "random_forest",
    {"modelo": "RandomForest", "config": "defecto"},
)

print(resultados.keys())
print(resultados["random_forest"][2].mean().round(4))

In [0]:
from sklearn.ensemble import GradientBoostingClassifier

resultados["gradient_boosting"] = entrenar_y_registrar(
    GradientBoostingClassifier(random_state=RANDOM_SEED),
    "gradient_boosting",
    {"modelo": "GradientBoosting", "config": "defecto"},
)

print(resultados.keys())
print(resultados["gradient_boosting"][2].mean().round(4))

In [0]:
from xgboost import XGBClassifier

resultados["xgboost"] = entrenar_y_registrar(
    XGBClassifier(random_state=RANDOM_SEED),
    "xgboost",
    {"modelo": "XGBoost", "config": "defecto"},
)

print(resultados.keys())
print(resultados["xgboost"][2].mean().round(4))

In [0]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.pr_auc_cv_media DESC"],
)

display(runs[["tags.mlflow.runName", "metrics.pr_auc_cv_media", "metrics.pr_auc_cv_std"]])

In [0]:
base = resultados["logreg_base"][2]

for nombre, (run_id, uri, scores) in resultados.items():
    diferencias = scores - base
    print(nombre, diferencias.round(3), diferencias.mean().round(4))

In [0]:
from sklearn.metrics import average_precision_score

run_id_ganador, uri_ganador, _ = resultados["logreg_base"]

modelo_ganador = mlflow.pyfunc.load_model(uri_ganador)
proba_test = np.asarray(modelo_ganador.predict(X_test))[:, 1]
pr_auc_test = average_precision_score(y_test, proba_test)

with mlflow.start_run(run_id=run_id_ganador):
    mlflow.log_metric("pr_auc_test", pr_auc_test)

print(round(pr_auc_test, 4))

## SPA:
**Elección del modelo**

Se compararon cuatro modelos con la misma partición, los mismos cinco trozos de validación cruzada, el mismo pipeline y su configuración por defecto. La métrica de decisión es la PR-AUC media en validación cruzada sobre entrenamiento.

| Modelo | PR-AUC media | Desviación | Diferencia con el baseline |
|---|---|---|---|
| Gradient Boosting | 0,6649 | 0,0090 | +0,0136 |
| Regresión logística | 0,6513 | 0,0208 | — |
| XGBoost | 0,6117 | 0,0183 | −0,0396 |
| Random Forest | 0,6107 | 0,0182 | −0,0407 |

Aplicando la regla del ruido, la mejora del Gradient Boosting queda por debajo de 0,02 y se considera un empate técnico. Random Forest y XGBoost pierden en los cinco trozos, de modo que son peores de forma consistente con su configuración por defecto.

Ante el empate se elige la regresión logística: es más sencilla, más ligera, está mejor calibrada, algo importante porque el modelo servirá probabilidades, y se explica con coeficientes.

**Límite conocido.** La comparación se hizo sin ajustar hiperparámetros. Los modelos de árboles dependen mucho de su configuración: el Random Forest pasó de 0,661 a 0,611 solo por volver a los valores por defecto. Un ajuste igual para todos queda como mejora posterior, con el modelo ajustado registrado como challenger frente a este.